# The Soundtrack of Late-Night Hustle: What I Listen to While Driving Uber

What do you listen to when you’re alone at 2 AM, driving strangers across the city?



In [16]:
import pandas as pd

rides = pd.read_csv('./data/rides.csv', parse_dates=['ride_start', 'ride_end'], dtype={'pickup_zip': 'str', 'dropoff_zip': 'str'})
rides = rides.query('ride_type not in ("Connect Express", "Connect Saver", "Delivery", "Package Express")')
rides = rides.sort_values('ride_start')
rides['ride_id'] = rides.index

songs = pd.read_csv('./data/spotify_enriched.csv', parse_dates=['time_start', 'time_end'])

ride_to_songs = {}
for i, ride in rides.iterrows():
    ride_start = ride['ride_start']
    ride_end = ride['ride_end']
    
    # Boolean mask for songs that satisfy:
    # ride_start < time_end < ride_end OR ride_start < time_start < ride_end
    mask = ((songs['time_start'] > ride_start) & (songs['time_start'] < ride_end)) | \
           ((songs['time_end'] > ride_start) & (songs['time_end'] < ride_end))
    matching_songs = songs[mask].copy()
    
    ride_id = ride.get('ride_id', i)
    ride_to_songs[ride_id] = matching_songs    

In [17]:
song_counts = {ride_id: len(songs) for ride_id, songs in ride_to_songs.items()}
ride_explicit_counts = {ride_id: songs['explicit'].sum() for ride_id, songs in ride_to_songs.items()}
popularity_total = {ride_id: songs['popularity'].sum() for ride_id, songs in ride_to_songs.items()}
rides['song_count'] = rides['ride_id'].map(song_counts).fillna(0).astype(int)
rides['explicit_count'] = rides['ride_id'].map(ride_explicit_counts).fillna(0).astype(int)
rides['popularity_total'] = rides['ride_id'].map(popularity_total).fillna(0).astype(int)

In [22]:
print(rides.groupby('explicit_count')['ride_type'].count())
print(rides.groupby('explicit_count')['tip'].sum())

explicit_count
0     2456
1      847
2      410
3      226
4      123
5       77
6       40
7       35
8       14
9       13
10       6
11       7
12       3
13       3
14       1
15       2
16       1
17       1
19       1
22       1
24       1
30       1
43       1
Name: ride_type, dtype: int64
explicit_count
0     3453.56
1     1217.43
2      669.79
3      387.37
4      183.29
5      184.36
6       94.10
7       59.88
8       43.98
9       35.35
10       9.12
11       8.05
12       7.00
13      10.44
14       5.18
15       5.00
16       0.00
17       1.00
19       0.00
22       4.38
24       0.00
30       3.40
43       0.00
Name: tip, dtype: float64
